In [40]:
import pyreadr
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_colwidth', None)


## Variable Descriptions:

In [3]:
# field_descriptions:

field_descrip_r = pyreadr.read_r('data/field_descriptions.rda')
#print(field_descrip_r)
field_descrip = pd.DataFrame(field_descrip_r['field_descriptions'])
field_descrip

,Field,Description
0,play_id,Numeric play id that when used with game_id and drive provides the unique identifier for a single play.
1,game_id,Ten digit identifier for NFL game.
2,old_game_id,Legacy NFL game ID.
3,home_team,String abbreviation for the home team.
4,away_team,String abbreviation for the away team.
...,...,...
367,xyac_median_yardage,Median expected yards after the catch based on where the ball was caught.
368,xyac_success,Probability play earns positive EPA (relative to where play started) based on where ball was caught.
369,xyac_fd,Probability play earns a first down based on where the ball was caught.
370,xpass,Probability of dropback scaled from 0 to 1.


In [4]:
# nfl_stats_variables:

nfl_stats_variables_r = pyreadr.read_r('data/nfl_stats_variables.rda')
#print(nfl_stats_variables_r)
nfl_stats_variables = pd.DataFrame(nfl_stats_variables_r['nfl_stats_variables'])
nfl_stats_variables

,variable,description
0,player_id,GSIS player ID. Available if stat_type = 'player'.
1,player_name,Short player name as listed in play-by-play data. Please keep in mind that this name is not always unique for one player and can change from season to season and sometimes even within a season. Do not group by this variable. Available if stat_type = 'player'.
2,player_display_name,Full name of player. Available if stat_type = 'player'.
3,position,Position of player. Available if stat_type = 'player'.
4,position_group,Position group of player. Available if stat_type = 'player'.
...,...,...
113,gwfg_blocked,Game winning field goal attempts blocked by opponent.
114,gwfg_distance,Distance of game winning field goal attempt. Available if summary_level = 'week'.
115,gwfg_distance_list,Distances of game winning field goal attempts. Available if summary_level = 'season'.
116,fantasy_points,Standard fantasy points.


In [5]:
# stat_ids:
stat_ids_r = pyreadr.read_r('data/stat_ids.rda')
#print(stat_ids_r)
stat_ids = pd.DataFrame(stat_ids_r['stat_ids'])
stat_ids

,stat_id,name,comment
0,1,Rushing Yards - Minus,Used in addition to the other Rushing stats so that total minus rushing yards can be calculated. THIS STAT IS NOT IN USE.
1,2,Punt Blocked (Offense),"Punt was blocked. A blocked punt is a punt that is touched behind the line of scrimmage, and is recovered, or goes out of bounds, behind the line of scrimmage. If the impetus of the punt takes it beyond the line of scrimmage, it is not a ""blocked punt."" This stat is used exclusively of the PU, PU_EZ, and PU_TB stats."
2,3,1st Down Rushing,A first down or TD occurred due to a rush.
3,4,1st Down Passing,A first down or TD occurred due to a pass.
4,5,1st Down Penalty,A first down or TD occurred due to a penalty. A play can have a first down from a pass or rush and from a penalty.
...,...,...,...
114,404,Defensive Two Point Conversions,"Defender intercepted or recovered an offensive fumble or (NFL only, recovered a blocked kick) during a two point conversion try and returned the ball for two defensive extra points."
115,405,Defensive Extra Point Attempts,(NCAA) Defender recovered a blocked extra point try.
116,406,Defensive Extra Point Conversions,(NCAA) Defender recovered a blocked extra point try and returned the ball for two defensive extra points.
117,410,Kickoff Length,"Kickoff and length of kick. Includes end zone yards for all kicks into the end zone, including kickoffs ending in a touchback."


## CSVs:

### 2019 data:

In [6]:
# Show all columns and their dtypes
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2019.dtypes)
data2019 = pd.read_csv('data/pbp_2019.csv', low_memory=False)
data2019['season'] = 2019
data2019.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,NaN,NaN,NaN,...,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,51,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,0,-1.658763,NaN,NaN,NaN,NaN,NaN,0.486799,51.320082
3,79,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,0,-0.538914,NaN,NaN,NaN,NaN,NaN,0.639994,-63.999379
4,100,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,0,0.142138,NaN,NaN,NaN,NaN,NaN,0.933516,6.648362


### 2020 data:

In [7]:
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2020.dtypes)
data2020 = pd.read_csv('data/pbp_2020.csv', low_memory=False)
data2020['season'] = 2020
data2020.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2020_01_ARI_SF,2020091311,SF,ARI,REG,1,NaN,NaN,NaN,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39,2020_01_ARI_SF,2020091311,SF,ARI,REG,1,SF,home,ARI,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,54,2020_01_ARI_SF,2020091311,SF,ARI,REG,1,SF,home,ARI,...,0,1,1.294838,0.50337,4.275047,2.0,0.619306,0.239695,0.515058,48.494154
3,93,2020_01_ARI_SF,2020091311,SF,ARI,REG,1,SF,home,ARI,...,0,1,0.857214,NaN,NaN,NaN,NaN,NaN,0.413357,-41.335732
4,118,2020_01_ARI_SF,2020091311,SF,ARI,REG,1,SF,home,ARI,...,0,1,-0.454665,NaN,NaN,NaN,NaN,NaN,0.446920,-44.692025


### 2021 data:

In [8]:
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2021.dtypes)
data2021 = pd.read_csv('data/pbp_2021.csv', low_memory=False)
data2021['season'] = 2021
data2021.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2021_01_ARI_TEN,2021091207,TEN,ARI,REG,1,NaN,NaN,NaN,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40,2021_01_ARI_TEN,2021091207,TEN,ARI,REG,1,TEN,home,ARI,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,55,2021_01_ARI_TEN,2021091207,TEN,ARI,REG,1,TEN,home,ARI,...,0,1,-1.399805,NaN,NaN,NaN,NaN,NaN,0.491433,-49.143299
3,76,2021_01_ARI_TEN,2021091207,TEN,ARI,REG,1,TEN,home,ARI,...,0,1,0.032412,1.165133,5.803177,4.0,0.896654,0.125098,0.697346,30.265415
4,100,2021_01_ARI_TEN,2021091207,TEN,ARI,REG,1,TEN,home,ARI,...,0,1,-1.532898,0.256036,4.147637,2.0,0.965009,0.965009,0.978253,2.174652


### 2022 data:

In [9]:
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2022.dtypes)
data2022 = pd.read_csv('data/pbp_2022.csv', low_memory=False)
data2022['season'] = 2022
data2022.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2022_01_BAL_NYJ,2022091107,NYJ,BAL,REG,1,NaN,NaN,NaN,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,43,2022_01_BAL_NYJ,2022091107,NYJ,BAL,REG,1,NYJ,home,BAL,...,0,1,-0.443521,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68,2022_01_BAL_NYJ,2022091107,NYJ,BAL,REG,1,NYJ,home,BAL,...,0,1,1.468819,NaN,NaN,NaN,NaN,NaN,0.440373,-44.037291
3,89,2022_01_BAL_NYJ,2022091107,NYJ,BAL,REG,1,NYJ,home,BAL,...,0,1,-0.492192,0.727261,6.988125,6.0,0.60693,0.227598,0.389904,61.009598
4,115,2022_01_BAL_NYJ,2022091107,NYJ,BAL,REG,1,NYJ,home,BAL,...,0,1,-0.325931,NaN,NaN,NaN,NaN,NaN,0.443575,-44.357494


### 2023 data:

In [10]:
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2023.dtypes)
data2023 = pd.read_csv('data/pbp_2023.csv', low_memory=False)
data2023['season'] = 2023
data2023.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,NaN,NaN,NaN,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,55,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,0,1,-0.336103,NaN,NaN,NaN,NaN,NaN,0.515058,-51.505846
3,77,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,0,1,0.703308,0.340652,3.328642,1.0,0.996628,0.583928,0.661106,33.889407
4,102,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,0,1,0.469799,NaN,NaN,NaN,NaN,NaN,0.196065,-19.606467


### 2024 data:

In [11]:
pd.set_option('display.max_rows', None)  # Show all rows (one per column)
#print(data2024.dtypes)
data2024 = pd.read_csv('data/pbp_2024.csv', low_memory=False)
data2024['season'] = 2024
data2024.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1,2024_01_ARI_BUF,2024090801,BUF,ARI,REG,1,NaN,NaN,NaN,...,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40,2024_01_ARI_BUF,2024090801,BUF,ARI,REG,1,ARI,away,BUF,...,0,0,0.257819,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,61,2024_01_ARI_BUF,2024090801,BUF,ARI,REG,1,ARI,away,BUF,...,0,0,-0.200602,NaN,NaN,NaN,NaN,NaN,0.456761,-45.676103
3,83,2024_01_ARI_BUF,2024090801,BUF,ARI,REG,1,ARI,away,BUF,...,0,0,2.028874,1.345418,9.321221,8.0,0.509778,0.363807,0.576656,42.334431
4,108,2024_01_ARI_BUF,2024090801,BUF,ARI,REG,1,ARI,away,BUF,...,0,0,0.754242,0.882798,5.783560,4.0,0.668478,0.255140,0.426443,57.355690


## **Goal:** Train a predictive model that, given game situation data, predicts whether a team should go for it on 4th down.

In [35]:
all_data = pd.concat([data2019, data2020, data2021, data2022, data2023, data2024], ignore_index=True)
#all_data.head()
#all_data.to_csv('data/pbp_2019_2024_merged.csv', index=False)

#relevant fourth down plays (pass, run, field-goal, punt)
fourth_down_plays = all_data[all_data['down']==4]
valid_play_types = ['pass', 'run', 'field_goal', 'punt']
relevent_fourths = fourth_down_plays[
    (fourth_down_plays['play_type'].isin(valid_play_types)) &
    (fourth_down_plays['ydstogo'].notna()) &
    (fourth_down_plays['play_type'].notna()) &
    (fourth_down_plays['play'].notna())]
relevent_fourths = relevent_fourths.copy()

def decision_col(row):
    if row['play_type'] in ['pass', 'run']:
        return 'go'
    elif row['play_type'] == 'punt':
        return 'punt'
    elif row['play_type'] == 'field_goal':
        return 'field_goal'
    else:
        return 'other'

relevent_fourths.loc[:, 'decision'] = relevent_fourths.apply(decision_col, axis=1)

relevent_fourths.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,decision
5,121,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,-4.034299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,punt
33,813,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,-0.022631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,punt
43,1039,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,MIN,home,ATL,...,0,-0.398995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,punt
75,1861,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0,-0.353419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,punt
82,1994,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,MIN,home,ATL,...,0,-0.635538,NaN,NaN,NaN,NaN,NaN,NaN,NaN,punt


### Encoding variables:

In [37]:
y = relevent_fourths['decision']
feature_cols = [
    'ydstogo',
    'yardline_100',
    'qtr',
    'down',
    'score_differential',
    'posteam_timeouts_remaining',
    'defteam_timeouts_remaining',
    'half_seconds_remaining',
    'game_seconds_remaining',
    'home_team',
    'away_team',
    'posteam',
    'season']
X = relevent_fourths[feature_cols]
#print(X.isnull().sum()) #check

categorical_cols = ['home_team', 'away_team', 'posteam', 'season']
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

encoder = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numeric_cols)])
X_encoded = encoder.fit_transform(X)
# encoded_feature_names = encoder.get_feature_names_out()
# X_encoded_df = pd.DataFrame(X_encoded, columns=encoded_feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y)

### Logistic Regression:

In [38]:
logreg_model = LogisticRegression(max_iter=1000)
logreg_model.fit(X_train, y_train)

y_pred_logreg = logreg_model.predict(X_test)
print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_logreg))

Logistic Regression Results:
              precision    recall  f1-score   support

  field_goal       0.81      0.83      0.82      1186
          go       0.63      0.55      0.59       910
        punt       0.91      0.95      0.93      2638

    accuracy                           0.84      4734
   macro avg       0.79      0.77      0.78      4734
weighted avg       0.83      0.84      0.84      4734



### RandomForestClassifier:

In [41]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))

Random Forest Results:
              precision    recall  f1-score   support

  field_goal       0.88      0.92      0.90      1186
          go       0.84      0.66      0.74       910
        punt       0.93      0.97      0.95      2638

    accuracy                           0.90      4734
   macro avg       0.88      0.85      0.86      4734
weighted avg       0.90      0.90      0.90      4734



### Training model:

In [32]:
# Evaluate
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[ 984  181   21]
 [ 191  496  223]
 [  33  108 2497]]

Classification Report:
              precision    recall  f1-score   support

  field_goal       0.81      0.83      0.82      1186
          go       0.63      0.55      0.59       910
        punt       0.91      0.95      0.93      2638

    accuracy                           0.84      4734
   macro avg       0.79      0.77      0.78      4734
weighted avg       0.83      0.84      0.84      4734



## Results:

The Random Forest model demonstrated strong performance, achieving an overall accuracy of 90% on the test set. It predicted field goal decisions with high precision (88%), recall (92%), and an F1-score of 90%, indicating consistent and accurate identification. Punt decisions were also predicted very well, with precision of 93%, recall of 97%, and an F1-score of 95%. While the model performed well on “go for it” decisions with 84% precision, its recall was lower at 66%, suggesting some missed cases, resulting in an F1-score of 74%. In contrast, the baseline model with simpler preprocessing achieved a lower overall accuracy of 84%. It showed moderate performance on field goals (81% precision, 83% recall, 82% F1-score) and punts (91% precision, 95% recall, 93% F1-score), but struggled with “go for it” predictions, with precision at 63%, recall at 55%, and an F1-score of 59%. These results indicate that using Random Forest combined with proper preprocessing and feature encoding improves classification accuracy and better balances predictions across all decision types, especially enhancing the model’s ability to detect “go for it” plays.